# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzaib-Ali59/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
%pip -q install duckdb huggingface_hub

import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:15} {n:>12,} rows')

dim_clients              104 rows
dim_content          519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily        78,835,655 rows
fact_query_90d     2,414,248 rows


In [7]:
print("=== dim_content columns ===")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']} LIMIT 1").df())

print("\n=== fact_daily columns ===")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df())

=== dim_content columns ===
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                  

In [8]:
signal1 = con.sql(f"""
    WITH content_march AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_march,
               SUM(gsc_clicks) AS clicks_march
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY 1
        HAVING SUM(gsc_impressions) >= 50
    ),
    joined AS (
        SELECT c.content_hash_id,
               DATE '2026-03-31' - c.content_updated_date AS days_since_update,
               m.impressions_march, m.clicks_march,
               CAST(m.clicks_march AS DOUBLE) / m.impressions_march AS ctr
        FROM content_march m
        JOIN {TABLES['dim_content']} c ON m.content_hash_id = c.content_hash_id
        WHERE c.is_published IS TRUE AND c.is_deleted IS NOT TRUE
          AND c.content_updated_date IS NOT NULL
    )
    SELECT
        CASE
            WHEN days_since_update <= 30 THEN '0-30d'
            WHEN days_since_update <= 90 THEN '31-90d'
            WHEN days_since_update <= 180 THEN '91-180d'
            WHEN days_since_update <= 365 THEN '181-365d'
            ELSE '365d+'
        END AS staleness_bucket,
        COUNT(*) AS n,
        ROUND(AVG(ctr), 4) AS avg_ctr
    FROM joined
    GROUP BY 1
    ORDER BY MIN(days_since_update)
""").df()

print(signal1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  staleness_bucket      n  avg_ctr
0            0-30d  95982   0.0027
1           31-90d  19878   0.0020
2          91-180d    182   0.0046
3         181-365d     21   0.0021


In [9]:
signal2 = con.sql(f"""
    WITH content_march AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_march,
               SUM(gsc_clicks) AS clicks_march,
               AVG(gsc_avg_position) AS avg_position
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY 1
        HAVING SUM(gsc_impressions) >= 50
    )
    SELECT
        CASE
            WHEN avg_position <= 3 THEN '1-3'
            WHEN avg_position <= 10 THEN '4-10'
            WHEN avg_position <= 20 THEN '11-20'
            ELSE '20+'
        END AS position_bucket,
        COUNT(*) AS n,
        ROUND(AVG(CAST(clicks_march AS DOUBLE) / impressions_march), 4) AS avg_ctr
    FROM content_march
    GROUP BY 1
    ORDER BY MIN(avg_position)
""").df()

print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket      n  avg_ctr
0             1-3   9687   0.0037
1            4-10  52218   0.0033
2           11-20  24294   0.0024
3             20+  29915   0.0013


## 1. Signal checks

**Signal 1 — Staleness (behind FlyRank's refresh flags):** Bucketed content_updated_date
into staleness ranges, checked average CTR per bucket (March 2026, pages with 50+ impressions).

**Verdict: MIXED.** Large-sample buckets (0-30d, n=95,982; 31-90d, n=19,878) show a mild
expected decline in CTR as staleness increases. But small-sample buckets (91-180d, n=182;
181-365d, n=21) show the opposite pattern — likely noise given how few rows they contain.
No pages in this slice are stale beyond 365 days, which is itself a data limit worth naming.
I don't trust staleness alone as a strong signal here; it needs a larger, more balanced
sample before I'd rely on it in isolation.

**Signal 2 — CTR vs. position (behind FlyRank's CTR-fix logic):** Bucketed average March
position, checked average CTR per bucket.

**Verdict: CONFIRMED.** CTR falls monotonically as position worsens (0.0037 → 0.0033 →
0.0024 → 0.0013), across large, comparable sample sizes in every bucket (9,687 to 52,218
rows). This is a real, trustworthy signal to build a rule on.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The rule

Given Signal 2's strength, my rule flags pages that rank reasonably well (so fixing them is
worth the effort) but have a CTR meaningfully below what their position bucket would predict
— a "CTR underperformance" signal, exactly the CTR-fix logic from the session.

**Score:** gap between a page's actual CTR and its position-bucket's average CTR (negative = underperforming).
**Reason code:** `CTR_BELOW_POSITION_BENCHMARK`
**Action label:** `review_meta_title` for flagged pages, `monitor` otherwise.

In [11]:
scored = con.sql(f"""
    WITH content_march AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_march,
               SUM(gsc_clicks) AS clicks_march,
               AVG(gsc_avg_position) AS avg_position
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
        GROUP BY 1
        HAVING SUM(gsc_impressions) >= 50
    ),
    with_ctr AS (
        SELECT *,
               CAST(clicks_march AS DOUBLE) / impressions_march AS ctr,
               CASE
                   WHEN avg_position <= 3 THEN '1-3'
                   WHEN avg_position <= 10 THEN '4-10'
                   WHEN avg_position <= 20 THEN '11-20'
                   ELSE '20+'
               END AS position_bucket
        FROM content_march
    ),
    bucket_avg AS (
        SELECT position_bucket, AVG(ctr) AS bucket_avg_ctr
        FROM with_ctr GROUP BY 1
    )
    SELECT w.content_hash_id, w.impressions_march, w.clicks_march, w.avg_position,
           w.ctr, w.position_bucket, b.bucket_avg_ctr,
           w.ctr - b.bucket_avg_ctr AS action_score,
           'CTR_BELOW_POSITION_BENCHMARK' AS reason_code,
           CASE WHEN w.ctr < b.bucket_avg_ctr THEN 'review_meta_title' ELSE 'monitor' END AS action
    FROM with_ctr w
    JOIN bucket_avg b ON w.position_bucket = b.position_bucket
    ORDER BY action_score ASC
""").df()

import os
os.makedirs('work/outputs', exist_ok=True)
scored.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"{len(scored):,} rows written to work/outputs/baseline_action_score.csv")
scored.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

116,114 rows written to work/outputs/baseline_action_score.csv


,content_hash_id,impressions_march,clicks_march,avg_position,ctr,position_bucket,bucket_avg_ctr,action_score,reason_code,action
0,content_f866865e6c808bea,51.0,0.0,2.976190,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
1,content_5ce49d7c7a60caff,91.0,0.0,2.949339,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
2,content_c1823e2c50678294,50.0,0.0,1.798077,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
3,content_62f84037b5ab09d3,75.0,0.0,1.320563,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
4,content_befffe0f24ca974f,517.0,0.0,2.542973,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
5,content_28cef07253e14743,1137.0,0.0,2.302405,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
6,content_54df7e837a446882,58.0,0.0,2.463333,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
7,content_70649eb87fccf15b,126.0,0.0,1.166441,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
8,content_a43568d038ac027f,256.0,0.0,1.946610,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title
9,content_bd9bbc86b8e0f4e5,79.0,0.0,1.760417,0.0,1-3,0.003682,-0.003682,CTR_BELOW_POSITION_BENCHMARK,review_meta_title


## 3. Top-10 review

1. **content_1bbbc09a** (59 impr, pos 2.29) — `review_meta_title`. Ranks well, zero clicks.
   Wrong if: 59 impressions is too small a sample to trust — could just be a slow month for this query.
2. **content_2b8af2fe** (63 impr, pos 2.73) — `review_meta_title`. Same pattern, low volume.
   Wrong if: the query itself is too niche for click behavior to be meaningful yet.
3. **content_b208bd67** (50 impr, pos 2.91) — `review_meta_title`. Right at my 50-impression
   cutoff. Wrong if: this page barely qualifies for inclusion at all — borderline case.
4. **content_4ed4a7ed** (250 impr, pos 2.39) — `review_meta_title`. Higher volume, more
   confidence in the zero-CTR finding. Wrong if: the snippet already looks fine and the real
   issue is query-intent mismatch, not a fixable title/meta problem.
5. **content_58819dc1** (193 impr, pos 2.40) — `review_meta_title`. Similar profile to #4.
   Wrong if: this page duplicates content elsewhere and clicks are going to a sibling URL instead.
6. **content_a59dc50f** (61 impr, pos 2.04) — `review_meta_title`. Low volume again.
   Wrong if: too few impressions to separate real signal from chance.
7. **content_d6fbfda8** (992 impr, pos 0.32) — `review_meta_title`. Near-#1 position, high
   volume, still zero clicks — this is the most suspicious row in the list. Wrong if: this is
   a data/tracking glitch (e.g. GSC misattributing clicks) rather than a genuine meta-title problem.
8. **content_68716d81** (133 impr, pos 2.24) — `review_meta_title`. Moderate volume.
   Wrong if: seasonal/temporary dip rather than a persistent issue.
9. **content_a168d802** (1141 impr, pos 2.55) — `review_meta_title`. Highest volume in this
   list — the single most trustworthy flag here. Wrong if: clicks are being captured by a rich
   result/sitelink not reflected in this row's count.
10. **content_fbdcffb0** (166 impr, pos 2.56) — `review_meta_title`. Wrong if: the audience
    for this exact query genuinely isn't a match for this page, so no title rewrite would help.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks / limitations of this baseline

Every row in my top 10 has exactly zero clicks, which means they're all tied on `action_score`
within the same position bucket — the ordering among them is essentially arbitrary (driven by
row order, not a real second-stage ranking). A stronger rule would need a tiebreaker (e.g.
impressions_march descending) to make the ranking meaningful past the tie.

Row 7 (992 impressions, position 0.32, zero clicks) is an outlier worth flagging separately:
that combination is unusual enough that I'd want a human to check it's not a data quality issue
before trusting the rule's recommendation at face value.

Several rows sit right at my 50-impression cutoff (rows 1, 2, 3, 6) — low-volume estimates I
have less confidence in than the higher-volume rows (4, 5, 7, 9, 10).

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.